In [0]:
%sql
select * from my_catalog.bronze.control_table;

create table my_catalog.bronze.etl_control(
  table_name string not null,
  source_table string,
  target_table string,
  key_col string,
  Cond_cols string,
  LAST_MODIFIED date,
  ACTIVE int
);

INSERT INTO my_catalog.bronze.etl_control (
  table_name,
  source_table,
  target_table,
  key_col,
  cond_cols,
  last_modified,
  active
)
VALUES (
  'DIM_EMPLOYEE',
  'srctemp',
  'my_catalog.gold.DIM_EMPLOYEE',
  'EMP_ID',
  'SALARY,LOCATION,DEPT',
  current_date(),
  1
);


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
source_path='/Volumes/realtimeprojects/eteproject/scd/Employee_1.csv'

In [0]:
df=spark.read.format("csv").option("header","true").option("inferSchema","true").load(source_path)

In [0]:
#df.write.saveAsTable('my_catalog.silver.emp_src')
target_table="my_catalog.silver.emp_src"
df.write.format("delta").mode("overwrite").saveAsTable(target_table)

In [0]:
%sql
select * from my_catalog.silver.emp_src

In [0]:
%sql

drop table my_catalog.gold.DIM_EMPLOYEE;

create table my_catalog.gold.DIM_EMPLOYEE(
  EMP_ID string not null,
  NAME string,
  LOCATION string,
  SALARY int,
  DEPT string,
  START_DATE date,
  END_DATE date,
  IS_ACTIVE int
);


update my_catalog.gold.DIM_EMPLOYEE set start_date='2026-03-15' where EMP_ID in (1,2,3,4,5);


In [0]:
%sql
---SCD type -1
MERGE INTO my_catalog.gold.EMPLOYEE a
USING  my_catalog.silver.emp_src b
ON a.EMP_ID = b.EMP_ID
WHEN MATCHED THEN UPDATE SET
  a.NAME  = b.NAME,
  a.SALARY   = b.SALARY,
  a.LOCATION  = b.LOCATION,
  a.DEPT  = b.DEPT,
  a.START_DATE = current_timestamp(),
  a.END_DATE = to_date('9999-12-31'),
  a.IS_ACTIVE = 1
WHEN NOT MATCHED THEN INSERT (
  EMP_ID, NAME, LOCATION, SALARY, DEPT,START_DATE,END_DATE,IS_ACTIVE
) VALUES (
  b.EMP_ID, b.NAME, b.LOCATION, b.SALARY, b.DEPT, current_timestamp(), to_date('9999-12-31'), 1
);


In [0]:
df=spark.sql("""
select * from my_catalog.silver.emp_src
""")

In [0]:
df=spark.sql("""
select a.*,to_date(current_timestamp(),'yyyy-MM-dd') as START_DATE,to_date('9999-12-31','yyyy-MM-dd') as END_DATE,1 as IS_ACTIVE from my_catalog.silver.emp_src a
""")

In [0]:
df.createOrReplaceTempView("srctemp")

In [0]:
df.display()

In [0]:
%sql
--SCD type-2
MERGE INTO my_catalog.gold.DIM_EMPLOYEE a
USING srctemp b
ON a.EMP_ID = b.EMP_ID and a.salary<>b.salary
WHEN MATCHED THEN
  UPDATE SET IS_ACTIVE=0, END_DATE=to_date(current_timestamp(),'yyyy-MM-dd')
WHEN NOT MATCHED THEN
  INSERT (EMP_ID, NAME, LOCATION, SALARY, DEPT, START_DATE, END_DATE, IS_ACTIVE)
  VALUES (b.EMP_ID, b.NAME, b.LOCATION, b.SALARY, b.DEPT, to_date(current_timestamp(),'yyyy-MM-dd'), to_date('9999-12-31'), 1);

In [0]:
%sql
MERGE INTO my_catalog.gold.DIM_EMPLOYEE a
USING srctemp b
ON a.EMP_ID = b.EMP_ID 
WHEN NOT MATCHED and a.salary<>b.salary THEN
  INSERT (EMP_ID, NAME, LOCATION, SALARY, DEPT, START_DATE, END_DATE, IS_ACTIVE)
  VALUES (b.EMP_ID, b.NAME, b.LOCATION, b.SALARY, b.DEPT, to_date(current_timestamp(),'yyyy-MM-dd'), to_date('9999-12-31'), 1);

In [0]:
%sql
select * from my_catalog.gold.DIM_EMPLOYEE order by emp_id;

In [0]:
%sql
select * from my_catalog.silver.emp_src

In [0]:
%sql
--2
-- Insert a new active row for:
--  a) brand-new employees (no active target row), or
--  b) employees whose row we just expired in Step 1 (changed)
--INSERT INTO my_catalog.gold.DIM_EMPLOYEE
--  (EMP_ID, NAME, LOCATION, SALARY, DEPT, START_DATE, END_DATE, IS_ACTIVE)
SELECT
  src.EMP_ID,
  src.NAME,
  src.LOCATION,
  src.SALARY,
  src.DEPT,
  to_date(current_timestamp(),'yyyy-MM-dd')        AS START_DATE,
  to_date('9999-12-31')       AS END_DATE,
  1                           AS IS_ACTIVE
FROM my_catalog.silver.emp_src AS src
LEFT JOIN my_catalog.gold.DIM_EMPLOYEE AS tgt
  ON  tgt.EMP_ID    = src.EMP_ID
  AND tgt.IS_ACTIVE = 1
-- Insert if there is no active row (new employee), or if value changed (the active row was just expired)
WHERE tgt.EMP_ID IS NULL
   OR tgt.SALARY <> src.SALARY;

In [0]:
%sql
--type 2
--SCD type-2
MERGE INTO my_catalog.gold.DIM_EMPLOYEE a
USING srctemp b
ON a.EMP_ID = b.EMP_ID
WHEN MATCHED THEN
  UPDATE SET IS_ACTIVE=0, END_DATE=to_date(current_timestamp(),'yyyy-MM-dd')
WHEN NOT MATCHED THEN
  INSERT INTO my_catalog.gold.DIM_EMPLOYEE
   (EMP_ID, NAME, LOCATION, SALARY, DEPT, START_DATE, END_DATE, IS_ACTIVE)
  SELECT
  src.EMP_ID,
  src.NAME,
  src.LOCATION,
  src.SALARY,
  src.DEPT,
  to_date(current_timestamp(),'yyyy-MM-dd')        AS START_DATE,
  to_date('9999-12-31')       AS END_DATE,
  1                           AS IS_ACTIVE
FROM my_catalog.silver.emp_src AS src
LEFT JOIN my_catalog.gold.DIM_EMPLOYEE AS tgt
  ON  tgt.EMP_ID    = src.EMP_ID
  AND tgt.IS_ACTIVE = 1
-- Insert if there is no active row (new employee), or if value changed (the active row was just expired)
WHERE tgt.EMP_ID IS NULL
   OR tgt.SALARY <> src.SALARY